# Lab 06 Solution: Advanced Routing

**Goal:** Build sophisticated routing patterns including nested conditionals,
LLM-powered routing, and priority-based routing.

**What you'll learn:**
- Nested conditional edges (multi-level decision trees)
- LLM-powered routing (let the LLM choose the path)
- Priority routing (check urgency before category)
- Combining multiple routing strategies

Requires: `GROQ_API_KEY` in `.env`

In [ ]:
import os
from typing import TypedDict, Annotated
from operator import add
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

## TODO 1 Solution: Nested routing within tech

Full implementation with priority routing at level 1 and nested tech sub-routing at level 2.

In [ ]:
class NestedState(TypedDict):
    message: str
    category: str
    sub_category: str
    priority: str
    response: str
    audit: Annotated[list, add]

def llm_classify(state: NestedState) -> dict:
    prompt = (
        f"Classify this request into: hr, tech, finance, facilities\n"
        f"Also detect priority: urgent or normal\n"
        f"Request: {state['message']}\n"
        f"Reply:\nCATEGORY: ...\nPRIORITY: ..."
    )
    response = llm.invoke(prompt)
    text = response.content.lower()
    category = "general"
    priority = "normal"
    for line in text.split("\n"):
        if "category:" in line:
            cat = line.split(":")[-1].strip()
            if cat in ["hr", "tech", "finance", "facilities"]:
                category = cat
        elif "priority:" in line:
            pri = line.split(":")[-1].strip()
            if pri in ["urgent", "normal"]:
                priority = pri
    print(f"  [classify] {category}/{priority}")
    return {"category": category, "priority": priority, "audit": [f"Classified: {category}/{priority}"]}

def handle_urgent(state: NestedState) -> dict:
    resp = llm.invoke(f"URGENT response for: {state['message']}")
    return {"response": f"[URGENT] {resp.content.strip()}", "audit": ["Urgent handler"]}

def handle_hr(state: NestedState) -> dict:
    resp = llm.invoke(f"HR response for: {state['message']}")
    return {"response": resp.content.strip(), "sub_category": "hr", "audit": ["HR handler"]}

def handle_finance(state: NestedState) -> dict:
    resp = llm.invoke(f"Finance response for: {state['message']}")
    return {"response": resp.content.strip(), "sub_category": "finance", "audit": ["Finance handler"]}

In [ ]:
# Nested tech routing
def tech_sub_route(state: NestedState) -> str:
    msg = state["message"].lower()
    if any(w in msg for w in ["server", "database", "network", "down"]):
        return "infra"
    elif any(w in msg for w in ["code", "bug", "deploy", "api"]):
        return "dev"
    return "general_tech"

def handle_infra(state: NestedState) -> dict:
    resp = llm.invoke(f"Infrastructure support for: {state['message']}")
    return {"response": resp.content.strip(), "sub_category": "infrastructure", "audit": ["Infra handler"]}

def handle_dev(state: NestedState) -> dict:
    resp = llm.invoke(f"Development support for: {state['message']}")
    return {"response": resp.content.strip(), "sub_category": "development", "audit": ["Dev handler"]}

def handle_general_tech(state: NestedState) -> dict:
    return {"response": "Tech: General ticket created.", "sub_category": "general_tech", "audit": ["General tech"]}

def handle_general(state: NestedState) -> dict:
    return {"response": "Your request has been logged.", "sub_category": "general", "audit": ["General handler"]}

def tech_triage(state: NestedState) -> dict:
    return {"audit": ["Tech triage"]}

In [ ]:
def route_l1(state: NestedState) -> str:
    if state["priority"] == "urgent":
        return "urgent"
    mapping = {"hr": "hr", "tech": "tech_triage", "finance": "finance"}
    return mapping.get(state["category"], "general")

graph3 = StateGraph(NestedState)
graph3.add_node("classify", llm_classify)
graph3.add_node("urgent", handle_urgent)
graph3.add_node("hr", handle_hr)
graph3.add_node("finance", handle_finance)
graph3.add_node("tech_triage", tech_triage)
graph3.add_node("infra", handle_infra)
graph3.add_node("dev", handle_dev)
graph3.add_node("general_tech", handle_general_tech)
graph3.add_node("general", handle_general)

graph3.add_edge(START, "classify")

# Level 1 routing
graph3.add_conditional_edges("classify", route_l1, {
    "urgent": "urgent", "hr": "hr", "tech_triage": "tech_triage",
    "finance": "finance", "general": "general",
})

# Level 2: nested tech routing
graph3.add_conditional_edges("tech_triage", tech_sub_route, {
    "infra": "infra", "dev": "dev", "general_tech": "general_tech",
})

for node in ["urgent", "hr", "finance", "infra", "dev", "general_tech", "general"]:
    graph3.add_edge(node, END)

app = graph3.compile()

In [ ]:
print("classify -> [urgent | hr | finance | tech_triage->[infra|dev|general_tech] | general] -> END\n")

tests = [
    "URGENT: Everything is on fire!",
    "I need to apply for leave",
    "The production database is running slow",
    "Help me deploy the new API to staging",
    "How do I submit my expense report?",
    "General tech question about our stack",
]

for msg in tests:
    result = app.invoke({"message": msg, "audit": []})
    print(f"  '{msg[:45]}' -> [{result.get('sub_category', 'N/A')}] {result['response'][:40]}...")

## TODO 2 Solution: LLM confidence routing

If the LLM's confidence is below 5, route to a "clarify" node instead of guessing wrong.

In [ ]:
class ConfState(TypedDict):
    message: str
    category: str
    confidence: int
    response: str

def classify_with_confidence(state: ConfState) -> dict:
    prompt = (
        f"Classify: {state['message']}\n"
        f"Reply:\nCATEGORY: hr or tech or finance\nCONFIDENCE: 1-10"
    )
    response = llm.invoke(prompt)
    text = response.content.lower()
    category = "general"
    confidence = 5
    for line in text.split("\n"):
        if "category:" in line:
            cat = line.split(":")[-1].strip()
            if cat in ["hr", "tech", "finance"]:
                category = cat
        elif "confidence:" in line:
            try:
                confidence = int(line.split(":")[-1].strip().rstrip("."))
            except ValueError:
                confidence = 5
    print(f"  [classify] {category} (confidence: {confidence}/10)")
    return {"category": category, "confidence": confidence}

def clarify(state: ConfState) -> dict:
    return {"response": f"I'm not sure I understand. Could you provide more details about: '{state['message']}'?"}

def handle(state: ConfState) -> dict:
    return {"response": f"[{state['category'].upper()}] Your request has been routed."}

def route_with_confidence(state: ConfState) -> str:
    if state["confidence"] < 5:
        return "clarify"
    return "handle"

In [ ]:
g = StateGraph(ConfState)
g.add_node("classify", classify_with_confidence)
g.add_node("clarify", clarify)
g.add_node("handle", handle)
g.add_edge(START, "classify")
g.add_conditional_edges("classify", route_with_confidence, {"clarify": "clarify", "handle": "handle"})
g.add_edge("clarify", END)
g.add_edge("handle", END)
app2 = g.compile()

for msg in ["I need sick leave", "asdfghjkl", "Something about the thing"]:
    result = app2.invoke({"message": msg})
    print(f"  '{msg}' -> {result['response'][:60]}...")

## Key Takeaways

- **LLM routing:** let the LLM classify and choose the path
- **Priority routing:** check urgency BEFORE category
- **Nested routing:** chain conditional edges for sub-categories (tech_triage node)
- **Confidence routing:** low-confidence classifications go to "clarify" instead of wrong route
- Always include **fallback routes** for unexpected classifications